# Scientific Reports successor — Step 8

Reduced predictive law, one-to-three-mode kernel selection and strict held-out validation. This notebook consumes Step 7 and stops before constitutive robustness and claim locking.

In [ ]:
from pathlib import Path
import subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'successor/scirep-waveform-susceptibility'
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    repo_root = Path('/content/picoNewton')
    if not repo_root.exists(): subprocess.run(['git','clone','https://github.com/khalid-saqr/picoNewton.git',str(repo_root)],check=True)
    subprocess.run(['git','-C',str(repo_root),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(repo_root),'checkout','-B',BRANCH,f'origin/{BRANCH}'],check=True)
    study_root = Path('/content/drive/MyDrive/picoNewton_susceptibility')
else:
    repo_root = Path.cwd()
    while repo_root != repo_root.parent and not (repo_root/'picoNewton_v3').exists(): repo_root = repo_root.parent
    study_root = repo_root/'piconewton_susceptibility_outputs'
print({'repo_root':str(repo_root),'study_root':str(study_root)})


## Install the parent and successor packages

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_root/'picoNewton_v3')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',str(repo_root/'piconewton_susceptibility')+'[dev]'],check=True)


## Execute the publication reduction study

In [ ]:
step7_root = study_root/'step7_waveform_experiments'
step8_root = study_root/'step8_reduced_law'
step8_root.mkdir(parents=True,exist_ok=True)
subprocess.run(['piconewton-susceptibility-step8','--step7-root',str(step7_root),'--output',str(step8_root),'--profile','publication'],check=True)


## Inspect model selection and the selected law

In [ ]:
import json, pandas as pd
selection=pd.read_csv(step8_root/'model_selection.csv')
family=pd.read_csv(step8_root/'compact_law_family_summary.csv')
law=json.loads((step8_root/'reduced_law.json').read_text())
display(selection)
display(family)
print(json.dumps(law,indent=2))


## Verify Step 8 closure

In [ ]:
gate=json.loads((step8_root/'step8_gate.json').read_text())
manifest=json.loads((step8_root/'step8_manifest.json').read_text())
assert gate['passed'] and manifest['status']=='complete' and manifest['allowed_next_step']==9
assert (step8_root/'step8_reduced_law.npz').is_file()
print({'gate':gate['passed'],'selected_rank':manifest['selected_rank'],'next':manifest['allowed_next_step']})
